In [4]:
import sqlite3
import pandas as pd

# Carregando a base para treino 

conn = sqlite3.connect("..\data\credit_risk.db")
df= pd.read_sql("SELECT * from credito_integrado",conn)
print(df.dtypes)
print(df.head)

person_age                      int64
person_income                 float64
person_home_ownership             str
person_emp_length              object
loan_intent                       str
loan_grade                        str
loan_amnt                     float64
loan_int_rate                  object
loan_status                     int64
loan_percent_income           float64
cb_person_default_on_file         str
cb_person_cred_hist_length      int64
date                            int64
year                            int64
month                           int64
unnamed_0                         str
unnamed_1                         str
ipca                          float64
no                            float64
dez_93_100                    float64
no_1                          float64
n12                           float64
ipca_tipo                         str
mes_num                         int64
data_ref                          str
selic_media                   float64
dtype: objec

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score 
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [ ]:
features = [
    'person_age', 
    'person_income',
    'person_emp_length', 
    'loan_amnt',
    'loan_int_rate', 
    'loan_percent_income', 
    'cb_person_cred_hist_length',
    'cb_person_default_on_file'
]

target = 'loan_status'

In [7]:
# Tratamento

df_model_gb = df[features + [target]].copy()

# converter Y/N -> 1/0 
df_model_gb['cb_person_default_on_file'] = df_model_gb['cb_person_default_on_file'].map({'Y': 1, 'N': 0})

# garantir numérico 
for col in df_model_gb.columns:
    df_model_gb[col] = pd.to_numeric(df_model_gb[col], errors = 'coerce')

df_model_gb.dropna(inplace=True)

# Binarios
df_model_gb = df_model_gb[df_model_gb[target].isin([0, 1])]
df_model_gb[target] = df_model_gb[target].astype(int)

print(df_model_gb.shape)
df_model_gb.head()

(28638, 9)


,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,cb_person_default_on_file,loan_status
0,22,59000.0,123.0,35000.0,16.02,0.59,3,1,1
1,21,9600.0,5.0,1000.0,11.14,0.10,2,0,0
2,25,9600.0,1.0,5500.0,12.87,0.57,3,0,1
3,23,65500.0,4.0,35000.0,15.23,0.53,2,0,1
4,24,54400.0,8.0,35000.0,14.27,0.55,4,1,1


In [8]:
# Split

X_gb = df_model_gb[features]
y_gb = df_model_gb[target]

X_train_gb, X_test_gb, y_train_gb, y_test_gb = train_test_split(
    X_gb, y_gb, 
    test_size= 0.2,
    random_state=42, 
    stratify=y_gb
)

print("Treino:", X_train_gb.shape, y_train_gb.shape)
print("Teste:", X_test_gb.shape, y_test_gb.shape)

Treino: (22910, 8) (22910,)
Teste: (5728, 8) (5728,)


In [9]:
# Modelo 

modelo_gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

In [10]:
# Validacao (cross-val)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores= cross_val_score(
    modelo_gb,
    X_train_gb,
    y_train_gb, 
    cv=cv,
    scoring='roc_auc'
)

print(f"AUC-ROC CV Gradient Boosting: {scores.mean():.4f} +- {scores.std():.4f}")

AUC-ROC CV Gradient Boosting: 0.9025 +- 0.0046


In [11]:
# Treinamento Final

modelo_gb.fit(X_train_gb, y_train_gb)
print("Deu bom! :)🎆")

Deu bom! :)🎆


In [12]:
# Teste

y_pred_gb = modelo_gb.predict(X_test_gb)
y_proba_gb = modelo_gb.predict_proba(X_test_gb)[:, 1]

auc_gb = roc_auc_score(y_test_gb, y_proba_gb)

print(f"\n AUC-ROC Test: {auc_gb:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_gb, y_pred_gb))


 AUC-ROC Test: 0.8998

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.95      0.92      4487
           1       0.76      0.61      0.67      1241

    accuracy                           0.87      5728
   macro avg       0.83      0.78      0.80      5728
weighted avg       0.87      0.87      0.87      5728



In [13]:
# Matriz de confusão + relevancia

cm_gb = confusion_matrix(y_test_gb, y_pred_gb)
print("Matriz de confusão - Gradient Boosting")
print(cm_gb)

importancias = pd.Series(
    modelo_gb.feature_importances_, 
    index=features
).sort_values(ascending=False)

print("\nImportância das variáveis:")
print(importancias)

Matriz de confusão - Gradient Boosting
[[4248  239]
 [ 489  752]]

Importância das variáveis:
loan_percent_income           0.380449
loan_int_rate                 0.319809
person_income                 0.199197
person_emp_length             0.050174
loan_amnt                     0.027103
person_age                    0.013160
cb_person_cred_hist_length    0.008887
cb_person_default_on_file     0.001220
dtype: float64


In [14]:
# Resumo 

print("Resumo do modelo")
print(f"AUC-ROC CV: {scores.mean():.4f} +- {scores.std():.4f}")
print(f"AUC-ROC Test: {auc_gb:.4f}")

Resumo do modelo
AUC-ROC CV: 0.9025 +- 0.0046
AUC-ROC Test: 0.8998


In [1]:
import joblib

joblib.dump(modelo_gb, "modelo_credito.pkl")
print("Modelo salvo como 'modelo_credito.pkl'")


NameError: name 'modelo_gb' is not defined

In [20]:
import numpy as np

probas = modelo_gb.predict_proba(X_test_gb)[:, 1] * 100

limites = np.percentile(probas, [10, 25, 40, 60, 75, 90])

print(f"Limites de risco (percentis): {limites[0]:.2f}, {limites[1]:.2f}, {limites[2]:.2f}, {limites[3]:.2f}, {limites[4]:.2f}, {limites[5]:.2f}")

Limites de risco (percentis): 2.03, 3.51, 5.48, 10.93, 26.08, 73.12
